# S02 — Data Wrangling and EDA
**LO2:** Implement data preprocessing pipelines including cleaning, feature engineering, scaling, and encoding using Python (pandas, scikit-learn).

Campus lab, 180 minutes. All data are synthetic; no model training in S02. Work in your own Colab copy. Audit → clean → split → EDA → engineer → fit preprocessing on training only.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
print(sys.version.split()[0], pd.__version__, sklearn.__version__)

## 1 — Generate intentionally imperfect, synthetic records
The `future_spend` outcome is not a valid feature for predicting from today's inputs.

In [ ]:
rng = np.random.default_rng(2026)
n = 60
df = pd.DataFrame({
 "record_id": [f"T{i:03d}" for i in range(n)],
 "age": rng.integers(19, 75, n).astype(float),
 "monthly_spend": np.round(rng.normal(110, 35, n).clip(5), 2).astype(object),
 "visits": rng.integers(0, 12, n),
 "channel": rng.choice(["web", "store", "phone"], n).astype(object),
 "region": rng.choice(["north", "south", "east"], n).astype(object),
 "future_spend": np.round(rng.normal(115, 25, n).clip(1), 2)
})
df.loc[3, "age"] = np.nan
df.loc[5, "age"] = 145
df.loc[9, "monthly_spend"] = "not recorded"
df.loc[14, "monthly_spend"] = None
df.loc[7, "channel"] = " WEB "
df.loc[11, "region"] = None
print(df.head(12).to_string(index=False))

## 2 — Data audit and documented deterministic cleaning
Ages outside 0–110 are treated as missing under an illustrative policy, not a universal truth. Numeric coercion creates additional missing values.

In [ ]:
print("Missing proportion:\n", df.isna().mean())
print("Category spelling:\n", df["channel"].value_counts(dropna=False))
print("Duplicate IDs:", df["record_id"].duplicated().sum())
clean = df.copy()
clean["monthly_spend"] = pd.to_numeric(clean["monthly_spend"], errors="coerce")
clean["age"] = clean["age"].where(clean["age"].between(0, 110))
for c in ["channel", "region"]:
    clean[c] = clean[c].astype("string").str.strip().str.lower().astype(object)
print("Missing after cleaning:\n", clean.isna().sum())

## 3 — Split before fitting preprocessing
A random split illustrates mechanics only. For time-series, sites or repeated subjects use an appropriate time/group split.

In [ ]:
X = clean.drop(columns=["record_id", "future_spend"])
y = clean["future_spend"]
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=0.2, random_state=42)
print("train:", X_train.shape, "holdout:", X_holdout.shape)
assert set(X_train.index).isdisjoint(set(X_holdout.index))

## 4 — Exploratory analysis of *training* observations
Histogram = distribution, bar chart = category frequency, scatter = association; none is causal proof.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14,3.5))
X_train["monthly_spend"].hist(bins=9, ax=ax[0])
ax[0].set(title="Monthly spend (train)", xlabel="Synthetic units", ylabel="Records")
X_train["channel"].value_counts().plot.bar(ax=ax[1])
ax[1].set(title="Channel frequencies (train)", xlabel="Channel", ylabel="Records")
ax[2].scatter(X_train["visits"], X_train["monthly_spend"])
ax[2].set(title="Visits vs spend (train)", xlabel="Visits", ylabel="Synthetic units")
plt.tight_layout()
plt.show()
print(X_train.describe(numeric_only=True).round(2))

## 5 — Feature engineering with division-by-zero protection
A zero-visit record has undefined spend-per-visit, not necessarily a value of zero.

In [ ]:
def engineer(frame):
    result = frame.copy()
    result["spend_per_visit"] = result["monthly_spend"].div(result["visits"].replace(0, np.nan))
    return result
X_train_fe = engineer(X_train)
X_holdout_fe = engineer(X_holdout)
assert not np.isinf(X_train_fe["spend_per_visit"].dropna().to_numpy(dtype=float)).any()

## 6 — Leakage-safe scikit-learn pipelines
Fit imputation, scaling and category vocabulary *once* on training rows. Apply unchanged to holdout. This produces a feature matrix without fitting a regression/classification model.

In [ ]:
numeric = ["age", "monthly_spend", "visits", "spend_per_visit"]
categorical = ["channel", "region"]
prep = ColumnTransformer([
 ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric),
 ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical)
])
train_features = prep.fit_transform(X_train_fe)
holdout_features = prep.transform(X_holdout_fe)
print("Shapes:", train_features.shape, holdout_features.shape)
print("Names:", prep.get_feature_names_out())
print("Training medians:", prep.named_transformers_["numeric"].named_steps["impute"].statistics_)
assert train_features.shape[1] == holdout_features.shape[1]

## 7 — Unseen holdout category
A new region must not cause refitting. `handle_unknown="ignore"` keeps output width stable.

In [ ]:
unseen = X_holdout_fe.iloc[[0]].copy()
unseen["region"] = "never-seen-before"
encoded = prep.transform(unseen)
print(encoded.shape)
assert encoded.shape[1] == train_features.shape[1]

## Independent practice — leave answers in your private copy
1. Find and justify three cleaning decisions. Explain possible bias from deleting missing rows.
2. Compare the effects of treating age 145 as plausible vs missing.
3. Explain why imputing or scaling on the whole dataset before splitting leaks information.
4. Introduce a new channel in holdout; describe why the encoder does not refit.
5. Explain why repeated observations per customer might require a group-based split.
6. Describe three plots, their axes/units, one possible alternative explanation, and one limitation.

In [ ]:
# Student workspace — do not commit completed answers or private data.